# Re-training and evaluating the YOLOv8n road-hazard detector

Pipeline originally built by my team. This notebook is my re-training and evaluation.

- Dataset: Roboflow Sinkhole_Vision_Fusion v1 (CC BY 4.0), 187 images.
- Result: mAP50 0.91 on the test split (15 images, 16 crack boxes). The validation and test splits contain no Pothole boxes.

In [1]:
!pip -q install ultralytics roboflow

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_KEY")
project = rf.workspace("xxx-r4bdj").project("sinkhole_vision_fusion")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Sinkhole_Vision_Fusion-1 in yolov8:: 100%|██████████| 386/386 [00:00<00:00, 6586.15it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640)

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Sinkhole_Vision_Fusion-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e7da57087c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [4]:
import glob, collections
for s in ["train", "valid", "test"]:
    c = collections.Counter()
    for f in glob.glob(f"{dataset.location}/{s}/labels/*.txt"):
        for line in open(f):
            if line.strip():
                c[int(line.split()[0])] += 1
    n = len(glob.glob(f"{dataset.location}/{s}/images/*"))
    print(s, "images:", n, "boxes per class (0=Crack, 1=Pothole):", dict(c))

train images: 142 boxes per class (0=Crack, 1=Pothole): {0: 114, 1: 47}
valid images: 30 boxes per class (0=Crack, 1=Pothole): {0: 31}
test images: 15 boxes per class (0=Crack, 1=Pothole): {0: 16}


In [5]:
best = YOLO("/content/runs/detect/train/weights/best.pt")
m = best.val(data=f"{dataset.location}/data.yaml", split="test")
print("mAP50:", m.box.map50, "mAP50-95:", m.box.map)
print("precision:", m.box.mp, "recall:", m.box.mr)
print("classes evaluated:", m.box.ap_class_index, "AP50 per class:", m.box.ap50)

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1297.5±471.8 MB/s, size: 41.6 KB)
val: Scanning /content/Sinkhole_Vision_Fusion-1/test/labels... 15 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 15/15 1.5Kit/s 0.0s
val: New cache created: /content/Sinkhole_Vision_Fusion-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.0it/s 0.2s
                   all         15         16      0.853      0.875       0.91      0.651
      Concentric_Crack         15         16      0.853      0.875       0.91      0.651
Speed: 0.4ms preprocess, 6.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.9096395705521473 mAP50-95: 0.651328403431646
precision: 0.853142838363004 recall: 0.875
classes evalua